# 06 — Démo : classification de lésions cutanées (prototype Gradio)
Charge l'ensemble final (**EfficientNetV2-S + CBAM** et **ConvNeXt-Tiny**), puis ouvre une page web où l'on dépose une image
dermoscopique et où l'on obtient :
- les **probabilités des 7 classes** (calibrées par température) ;
- la **décision au point clinique** (sensibilité mélanome ≥ 0.85, facteur choisi sur VAL) ;
- une **carte Grad-CAM** : les zones de l'image qui ont le plus pesé dans la décision.

**Pour l'essayer vous-même :** **Copy & Edit**, puis (panneau de droite) :
1. **Session options** → *Internet* : **On** ; *Accelerator* : **GPU T4** (conseillé ; le CPU marche mais ~10 s par image).
   Les poids (~1 Go) sont téléchargés automatiquement depuis
   [huggingface.co/RihemBousbih/skin-lesion-effnet-convnext](https://huggingface.co/RihemBousbih/skin-lesion-effnet-convnext).
2. *(Optionnel, pour avoir des images d'exemple)* **+ Add Input** → *Datasets* → `jpeg-isic2019-512x512` (cdeotte)
   et → *Notebooks* → `skincanerf-metadata-stacking` (liste des images du jeu de test).
3. **Run All** (dans l'éditeur, pas « Save Version »). Le lien `https://….gradio.live` apparaît dans la dernière cellule :
   il reste actif tant que la session Kaggle tourne (72 h maximum).

> ⚠️ Outil de recherche — **pas un dispositif médical**. Entraîné uniquement sur des images **dermoscopiques** (ISIC).

## 1 — Installation de Gradio

In [1]:
!pip install -q "gradio>=4.44" opencv-python-headless huggingface_hub

## 2 — Setup : localisation des modèles et de la calibration

In [2]:
import os, glob, json, time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import cv2

IMG_SIZE    = 384
EVAL_RESIZE = int(IMG_SIZE * 1.15)
CLASSES = ["akiec", "bcc", "bkl", "df", "nv", "mel", "vasc"]
MEL     = CLASSES.index("mel")
NOMS = {"akiec": "Kératose actinique / Bowen (akiec)",
        "bcc":   "Carcinome basocellulaire (bcc)",
        "bkl":   "Lésion kératosique bénigne (bkl)",
        "df":    "Dermatofibrome (df)",
        "nv":    "Nævus mélanocytaire (nv)",
        "mel":   "Mélanome (mel)",
        "vasc":  "Lésion vasculaire (vasc)"}
MALIN = {"mel": "maligne", "bcc": "maligne", "akiec": "précancéreuse / maligne in situ",
         "bkl": "bénigne", "df": "bénigne", "nv": "bénigne", "vasc": "bénigne"}


MODEL_REPO = "RihemBousbih/skin-lesion-effnet-convnext"   # poids publics sur Hugging Face


def find(name, required=True):
    # 1) dans les Inputs Kaggle si présents ; 2) sinon téléchargé depuis Hugging Face (Internet : On).
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if hits:
        return hits[0]
    if not required:
        raise FileNotFoundError(name)
    from huggingface_hub import hf_hub_download
    print(f"  {name} : téléchargement depuis huggingface.co/{MODEL_REPO} …")
    return hf_hub_download(MODEL_REPO, name)


EFF_PATH = find("effnetv2s_cbam_final.keras")
CX_PATH  = find("convnext_tiny_final_v2.keras")
CFG_PATH = find("run_config_final.json")
cfg = json.load(open(CFG_PATH))
T_FINAL = float(cfg["temperature"])
K_MEL   = float(cfg["facteur_mel"])
print("EfficientNet :", EFF_PATH)
print("ConvNeXt     :", CX_PATH)
print(f"Calibration  : T = {T_FINAL:.3f} | facteur mélanome k = {K_MEL:.3f} | modèles retenus = {cfg['modeles']}")
print("GPU :", tf.config.list_physical_devices("GPU") or "aucun (CPU)")

EfficientNet : /kaggle/input/notebooks/rihembousbih/skincanerf-phase3/effnetv2s_cbam_final.keras
ConvNeXt     : /kaggle/input/notebooks/rihembousbih/skincanerf-metadata-stacking/convnext_tiny_final_v2.keras
Calibration  : T = 0.673 | facteur mélanome k = 2.581 | modèles retenus = ['eff', 'cx2']
GPU : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 3 — Couches CBAM (nécessaires pour recharger EfficientNet) et prétraitement identique à l'évaluation

In [4]:
class ChannelAttention(layers.Layer):
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio
        self.gap = layers.GlobalAveragePooling2D()
        self.gmp = layers.GlobalMaxPooling2D()
        self.add = layers.Add()
        self.act = layers.Activation("sigmoid")
        self.mul = layers.Multiply()

    def build(self, input_shape):
        channels = int(input_shape[-1])
        hidden   = max(channels // self.ratio, 1)
        self.d1 = layers.Dense(hidden, activation="relu", kernel_initializer="he_normal", use_bias=True, dtype="float32")
        self.d2 = layers.Dense(channels, kernel_initializer="he_normal", use_bias=True, dtype="float32")
        self.d1.build((None, 1, 1, channels))
        self.d2.build((None, 1, 1, hidden))
        super().build(input_shape)

    def call(self, x):
        in_dtype = x.dtype
        x32 = tf.cast(x, tf.float32)
        c   = int(x.shape[-1])
        avg = tf.reshape(tf.reduce_mean(x32, axis=[1, 2]), (-1, 1, 1, c))
        mx  = tf.reshape(tf.reduce_max(x32, axis=[1, 2]), (-1, 1, 1, c))
        att = tf.sigmoid(self.d2(self.d1(avg)) + self.d2(self.d1(mx)))
        return tf.cast(x32 * att, in_dtype)

    def get_config(self):
        c = super().get_config(); c.update({"ratio": self.ratio}); return c


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
        self.mul = layers.Multiply()

    def build(self, input_shape):
        self.conv = layers.Conv2D(1, self.kernel_size, padding="same", activation="sigmoid",
                                  kernel_initializer="he_normal", use_bias=False, dtype="float32")
        self.conv.build((input_shape[0], input_shape[1], input_shape[2], 2))
        super().build(input_shape)

    def call(self, x):
        in_dtype = x.dtype
        x32 = tf.cast(x, tf.float32)
        avg = tf.reduce_mean(x32, axis=-1, keepdims=True)
        mx  = tf.reduce_max(x32, axis=-1, keepdims=True)
        att = self.conv(tf.concat([avg, mx], axis=-1))
        return tf.cast(x32 * att, in_dtype)

    def get_config(self):
        c = super().get_config(); c.update({"kernel_size": self.kernel_size}); return c


CUSTOM_OBJECTS = {"ChannelAttention": ChannelAttention, "SpatialAttention": SpatialAttention}


def shades_of_gray(img, p=6.0):
    flat  = tf.reshape(img, [-1, 3])
    illum = tf.pow(tf.reduce_mean(tf.pow(flat + 1e-6, p), axis=0), 1.0 / p)
    illum = illum / (tf.norm(illum) + 1e-6)
    img   = img / (illum * tf.sqrt(3.0) + 1e-6)
    return tf.clip_by_value(img, 0.0, 255.0)


def preprocess(rgb_uint8):
    # Même chaîne qu'à l'évaluation : resize 441 → center-crop 384 → Shades-of-Gray. Retourne (entrée modèle, image affichable).
    img = tf.image.resize(tf.cast(rgb_uint8, tf.float32), (EVAL_RESIZE, EVAL_RESIZE))
    off = (EVAL_RESIZE - IMG_SIZE) // 2
    img = tf.image.crop_to_bounding_box(img, off, off, IMG_SIZE, IMG_SIZE)
    shown = tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8).numpy()
    return shades_of_gray(img), shown


def tta_views(x):
    # 4 rotations × 2 flips, comme à l'évaluation.
    views = []
    for k in range(4):
        r = tf.image.rot90(x, k=k)
        views += [r, tf.image.flip_left_right(r)]
    return tf.stack(views)

print("Prétraitement défini.")

Prétraitement défini.


## 4 — Chargement des modèles
Les modèles ont été entraînés en *mixed precision* (float16). Pour l'inférence et Grad-CAM on les convertit en **float32**
(même poids, calcul plus stable, et le float16 est très lent sur CPU).

In [5]:
def to_float32(model):
    try:
        js = model.to_json().replace('"mixed_float16"', '"float32"')
        m32 = keras.models.model_from_json(js, custom_objects=CUSTOM_OBJECTS)
        m32.set_weights(model.get_weights())
        return m32
    except Exception as e:
        print("  (conversion float32 impossible, modèle gardé tel quel :", type(e).__name__, ")")
        return model


def gradcam_layer(model):
    # Carte de features 4D juste avant le pooling final (sortie CBAM pour EfficientNet, dernier bloc pour ConvNeXt).
    names = [l.name for l in model.layers]
    for cand in ("gap_c5", "gap"):
        if cand in names:
            return model.get_layer(cand).input
    return next(l.output for l in reversed(model.layers) if len(l.output.shape) == 4)


MODELS = {}
for name, path in [("EfficientNetV2-S + CBAM", EFF_PATH), ("ConvNeXt-Tiny", CX_PATH)]:
    t0 = time.time()
    m = keras.models.load_model(path, custom_objects=CUSTOM_OBJECTS, compile=False)
    m = to_float32(m)
    grad_model = keras.Model(m.inputs, [gradcam_layer(m), m.output])
    MODELS[name] = (m, grad_model)
    print(f"✅ {name} chargé en {time.time() - t0:.0f}s")

I0000 00:00:1790286310.477257      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790286310.480067      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


✅ EfficientNetV2-S + CBAM chargé en 12s
✅ ConvNeXt-Tiny chargé en 6s


## 5 — Prédiction, calibration, Grad-CAM et texte d'explication

In [6]:
def temp_scale(p, T):
    s = np.exp(np.log(np.clip(p, 1e-12, 1.0)) / T)
    return s / s.sum()


def gradcam(grad_model, x, cls):
    with tf.GradientTape() as tape:
        fmap, pred = grad_model(x[None], training=False)
        score = tf.math.log(pred[:, cls] + 1e-9)
    g = tape.gradient(score, fmap)
    w = tf.reduce_mean(g, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * fmap, axis=-1))[0].numpy()
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    return cam / (cam.max() + 1e-8)


def overlay(img_uint8, cam, alpha=0.45):
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)[:, :, ::-1]
    return np.uint8((1 - alpha) * img_uint8 + alpha * heat)


def analyse(image, use_tta=True):
    if image is None:
        return {}, "Déposez une image.", None
    t0 = time.time()
    x, shown = preprocess(np.asarray(image)[:, :, :3])
    batch = tta_views(x) if use_tta else x[None]

    per_model = {n: np.asarray(m(batch, training=False)).mean(0) for n, (m, _) in MODELS.items()}
    p_raw = np.mean(list(per_model.values()), axis=0)            # moyenne simple (choix fait sur VAL)
    p_cal = temp_scale(p_raw, T_FINAL)                           # probabilités calibrées (affichage)

    q = p_raw.copy(); q[MEL] *= K_MEL                            # point de fonctionnement clinique
    pred = int(q.argmax()); top = int(p_raw.argmax())

    cam = np.mean([gradcam(gm, x, pred) for _, gm in MODELS.values()], axis=0)
    img_cam = overlay(shown, cam)

    order = np.argsort(-p_cal)
    c1, c2 = CLASSES[order[0]], CLASSES[order[1]]
    lignes = [f"### Décision : **{NOMS[CLASSES[pred]]}** — lésion {MALIN[CLASSES[pred]]}"]
    if pred == MEL and top != MEL:
        lignes.append(f"⚠️ Le modèle place d'abord *{NOMS[CLASSES[top]]}*, mais la probabilité de mélanome "
                      f"({p_cal[MEL]:.0%}) dépasse le seuil de vigilance : par prudence, la lésion est signalée "
                      f"comme **mélanome suspecté**.")
    conf = p_cal[order[0]]
    niveau = "élevée" if conf >= 0.80 else ("modérée" if conf >= 0.55 else "faible")
    lignes.append(f"- **Confiance {niveau}** : {NOMS[c1]} à {p_cal[order[0]]:.0%}, "
                  f"puis {NOMS[c2]} à {p_cal[order[1]]:.0%}.")
    if conf < 0.55:
        lignes.append("- Le modèle **hésite** entre plusieurs classes : un avis spécialisé est indispensable.")
    accord = {n: CLASSES[int(p.argmax())] for n, p in per_model.items()}
    if len(set(accord.values())) == 1:
        lignes.append(f"- Les deux modèles de l'ensemble sont **d'accord** ({NOMS[list(accord.values())[0]]}).")
    else:
        lignes.append("- Les deux modèles **divergent** : " +
                      " ; ".join(f"{n} → {NOMS[c]}" for n, c in accord.items()) + ".")
    lignes.append("- **Carte de chaleur** : zones rouges = régions qui ont le plus pesé dans la décision "
                  "(Grad-CAM, moyenne des deux modèles). Elle montre *où* le modèle regarde, pas *pourquoi* médicalement.")
    lignes.append(f"\n<sub>Analyse en {time.time() - t0:.1f}s · TTA {'8 vues' if use_tta else 'désactivée'} · "
                  f"image redimensionnée et recadrée au centre (384 px), couleurs normalisées (Shades-of-Gray).</sub>")

    return {NOMS[c]: float(p_cal[i]) for i, c in enumerate(CLASSES)}, "\n".join(lignes), img_cam


# Contrôle rapide sur une image synthétique
_ = analyse(np.random.randint(0, 255, (450, 600, 3), dtype=np.uint8), use_tta=False)
print("✅ Pipeline de prédiction opérationnel.")

I0000 00:00:1790286332.998050      58 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2026-09-24 21:45:35.336805: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 21:45:35.491108: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 21:45:35.623218: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 21:45:41.387789: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 21:45:41.539508: E external/local_xla/xla/stream_

✅ Pipeline de prédiction opérationnel.


## 6 — Images d'exemple (tirées du **test**, jamais vues à l'entraînement) — optionnel

In [8]:
EXAMPLES = []
try:
    test_df = pd.read_csv(find("split_test.csv", required=False))
    ok = test_df[test_df["path"].map(os.path.exists)]
    import shutil
    os.makedirs("/kaggle/working/examples", exist_ok=True)       # Gradio ne sert que les fichiers de /kaggle/working ou /tmp
    for c in ["mel", "nv", "bcc", "bkl", "akiec", "vasc"]:
        sub = ok[ok["dx"] == c]
        if len(sub):
            src = sub.sample(1, random_state=0)["path"].iloc[0]
            dst = f"/kaggle/working/examples/{c}_{os.path.basename(src)}"
            shutil.copy(src, dst)
            EXAMPLES.append([dst, True])
    print(f"{len(EXAMPLES)} images d'exemple :", [os.path.basename(e[0]) for e in EXAMPLES])
except FileNotFoundError:
    pass
if not EXAMPLES:
    print("Pas d'images d'exemple (ajoutez le dataset jpeg-isic2019-512x512 en Input si vous en voulez).")

6 images d'exemple : ['mel_ISIC_0000511.jpg', 'nv_ISIC_0000235.jpg', 'bcc_ISIC_0060144.jpg', 'bkl_ISIC_0025099.jpg', 'akiec_ISIC_0060337.jpg', 'vasc_ISIC_0071024.jpg']


## 7 — Lancer la page web

In [10]:
import gradio as gr

AVERTISSEMENT = (
    "⚠️ **Outil de recherche — pas un dispositif médical.** Entraîné uniquement sur des images **dermoscopiques** "
    "(ISIC 2019). Une photo prise au téléphone est hors de son domaine : le résultat n'y est pas fiable. "
    "Aucune décision médicale ne doit être prise à partir de cet outil.")

with gr.Blocks(title="Classification de lésions cutanées") as demo:
    gr.Markdown("# Classification de lésions cutanées — démo de recherche\n"
                "Ensemble **EfficientNetV2-S + CBAM** et **ConvNeXt-Tiny** · 7 classes · "
                "test ISIC : accuracy 0.874, AUC 0.981 · validation externe PH2.")
    gr.Markdown(AVERTISSEMENT)
    with gr.Row():
        with gr.Column():
            inp = gr.Image(type="numpy", label="Image dermoscopique")
            tta = gr.Checkbox(value=True, label="TTA (8 vues : plus précis, un peu plus lent)")
            btn = gr.Button("Analyser", variant="primary")
            if EXAMPLES:
                gr.Examples(EXAMPLES, inputs=[inp, tta], label="Exemples (jeu de test)")
        with gr.Column():
            out_txt   = gr.Markdown()
            out_label = gr.Label(num_top_classes=7, label="Probabilités calibrées")
            out_cam   = gr.Image(label="Grad-CAM : zones déterminantes")
    btn.click(analyse, inputs=[inp, tta], outputs=[out_label, out_txt, out_cam])

demo.launch(share=True, allowed_paths=["/kaggle/working"])

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://7f3095e0ebc26cc5bb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-09-24 21:51:20.105234: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 21:51:20.243126: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
